# Spatially smoothed PCA global component

This notebook performs a brief visual sanity check of the spatially smoothed PCA global-component implementation.

We load an example SNIRF dataset, compute the global component, and compare:

1. raw data
2. spatially smoothed PCA global component
3. raw data minus global component

Note: this example dataset is channel-space data, so the spatial smoothing kernel is built from channel midpoint coordinates rather than vertex coordinates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

import cedalion
import cedalion.data
import cedalion.io
import cedalion.nirs
import cedalion.dataclasses as cdc
import cedalion.xrutils as xrutils

from cedalion.sigproc.physio import (
    get_spatial_smoothing_kernel,
    compute_Hglobal_from_PCA,
)

In [ ]:
rec = cedalion.data.get_snirf_test_data()
rec = rec[0]
rec

In [ ]:
ts = rec["amp"]
ts

In [ ]:
spatial_dim = cdc.get_spatial_dimension(ts)
spectral_dim = xrutils.other_dim(ts, spatial_dim, "time")

spatial_dim, spectral_dim

In [ ]:
geo3d = rec.geo3d

channel_pos = (
    geo3d.loc[ts.source].pint.dequantify()
    + geo3d.loc[ts.detector].pint.dequantify()
) / 2

vertices = channel_pos.values

vertices.shape

In [ ]:
W = get_spatial_smoothing_kernel(vertices, sigma_mm=80.0)

W.shape, W.min(), W.max()

In [ ]:
global_field = compute_Hglobal_from_PCA(
    ts,
    smoothing_kernel=W,
    spatial_dim=spatial_dim,
    spectral_dim=spectral_dim,
)

global_field

In [ ]:
residual = ts - global_field
residual

In [ ]:
spectral_value = ts[spectral_dim].values[0]

raw_2d = ts.sel({spectral_dim: spectral_value}).pint.dequantify()
global_2d = global_field.sel({spectral_dim: spectral_value}).pint.dequantify()
residual_2d = residual.sel({spectral_dim: spectral_value}).pint.dequantify()

n_show = min(10, ts.sizes[spatial_dim])

spectral_value, n_show

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(
    raw_2d.time,
    raw_2d.isel({spatial_dim: slice(0, n_show)}).transpose("time", spatial_dim),
)
plt.title(f"Raw data ({spectral_dim}={spectral_value})")
plt.xlabel("time")
plt.ylabel(str(ts.pint.units))
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(
    global_2d.time,
    global_2d.isel({spatial_dim: slice(0, n_show)}).transpose("time", spatial_dim),
)
plt.title(f"Spatially smoothed PCA global component ({spectral_dim}={spectral_value})")
plt.xlabel("time")
plt.ylabel(str(ts.pint.units))
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(
    residual_2d.time,
    residual_2d.isel({spatial_dim: slice(0, n_show)}).transpose("time", spatial_dim),
)
plt.title(f"Raw data minus global component ({spectral_dim}={spectral_value})")
plt.xlabel("time")
plt.ylabel(str(ts.pint.units))
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

raw_plot = raw_2d.transpose(spatial_dim, "time")
global_plot = global_2d.transpose(spatial_dim, "time")
residual_plot = residual_2d.transpose(spatial_dim, "time")

axes[0].imshow(raw_plot, aspect="auto")
axes[0].set_title("Raw data")
axes[0].set_xlabel("time index")
axes[0].set_ylabel(spatial_dim)

axes[1].imshow(global_plot, aspect="auto")
axes[1].set_title("Global component")
axes[1].set_xlabel("time index")

axes[2].imshow(residual_plot, aspect="auto")
axes[2].set_title("Raw - global")
axes[2].set_xlabel("time index")

plt.tight_layout()
plt.show()